# 05 · Optional: splice maps from RBP-ARK's windows table (no BAMs)

Last of six notebooks, and optional. Notebook 04 builds splice maps from eCLIP reads with rbp-maps, which needs the BAM files (3.7 GB). This notebook builds a coarser version from RBP-ARK's public windows table alone: for exons more included, less included or unchanged after knockdown, the share with a bound window (q ≤ 0.05, log2 enrichment ≥ 2) in each 50-nt bin within 300 nt of the exon, compared with background by Fisher's exact test. It runs in minutes and is a quick check that the read-based maps agree.

Uses ENCODE's released rMATS tables (as the screen in notebook 00 does) and the windows table that notebook 00 cached.

## Accessing RBP-ARK data through Globus

RBP-ARK keeps every Skipper output on one Globus collection. Public files can be downloaded straight over HTTPS with no login; bulk copies go through Globus.

| | |
| --- | --- |
| Globus collection | **RBP-ARK**, `9cd9be17-cbed-4b34-ab76-704b2fd6ac7b` ([open in the Globus web app](https://app.globus.org/file-manager?origin_id=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b&origin_path=%2F)) |
| HTTPS address of the same files | `https://g-f8db23.0ed28.75bc.data.globus.org/<path>` |

**Where things are.** `<version>/<run>` is the Skipper version and run folder. Since November 2026 every ENCODE3, ENCORE and ENCORE2 dataset on rbp-ark.com (gene search, dataset pages, BDBags) is Skipper `2.0.0/v2`, and so are these notebooks. Earlier runs (`1.99.0/v1`, `1.100.0/v1`) remain on the collection.

| Path on the collection | Contents |
| --- | --- |
| `/encode3/eclip/Skipper/<version>/<run>/output/` | ENCODE3 eCLIP datasets, named `<RBP>_<cell>_ENCSR…` |
| `/encore2/eclip/Skipper/<version>/<run>/output/` | ENCORE2 eCLIP datasets with 5000-series IDs (e.g. `QKI_K562_5007`) |
| `/encore/eclip/Skipper/<version>/<run>/output/` | ENCORE eCLIP datasets with 4000-series IDs |
| `/summary_data/` | summary tables (QC, window counts) |

Inside each `output/` folder, files are grouped by type, then named by dataset:

| Subfolder | Files |
| --- | --- |
| `reproducible_enriched_windows/` | `<dataset>.reproducible_enriched_windows.tsv.gz`, the enriched windows |
| `reproducible_enriched_re/` | enriched repetitive elements |
| `secondary_results/bams/dedup/genome_R2/` | deduplicated, read-2-only genome BAMs (ENCODE3) |
| `secondary_results/bams/dedup/genome/` | deduplicated genome BAMs (ENCORE) |
| `secondary_results/bams/raw/genome/` | alignments before deduplication |
| `secondary_results/bigwigs/scaled/plus/`, `.../minus/` | strand-specific signal tracks |
| `QC/`, `gene_sets/`, `homer/` | QC reports, gene set enrichment, motifs |

**Three ways to get files.**

1. **One dataset: BDBag.** Each dataset page on rbp-ark.com has a **Download BDBag** link. The bag's `fetch.txt` lists every file of that dataset with its HTTPS URL, size and path, and `manifest-sha256.txt` its checksum. Notebook 04 reads the same file list from the RBP-ARK API and downloads only the BAMs it needs.
2. **Many datasets: Globus web app.** Sign in at [app.globus.org](https://app.globus.org) with your institution or a free Globus ID, open the RBP-ARK collection (link above), select folders or files, and transfer them to a collection you can write to: your laptop via [Globus Connect Personal](https://www.globus.org/globus-connect-personal), or your institution's cluster or storage collection.
3. **Scripted: Globus CLI.**

```bash
pip install globus-cli
globus login                                   # once; opens a browser
RBPARK=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b
OUT=/encode3/eclip/Skipper/2.0.0/v2/output

# list what is there
globus ls "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/" | grep QKI

# copy one file to this computer (Globus Connect Personal must be running)
DEST=$(globus endpoint local-id)
globus transfer "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" \
    "$DEST:$HOME/rbpark/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" --label "QKI eCLIP"

# or copy a whole folder
globus transfer --recursive "$RBPARK:$OUT/reproducible_enriched_windows/" "$DEST:$HOME/rbpark/windows/"
globus task list                               # follow progress
```

If the CLI asks for extra consent the first time it touches the collection, it prints the exact `globus session consent ...` command to run. The Globus CLI commands are shown for reference and were not run in testing; the BDBag and HTTPS routes were. Please copy only what you need: the public HTTPS endpoint is shared, and bulk sweeps have taken it offline.

**In this notebook:** only RBP-ARK's windows table (data API), as in notebook 00.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

sys.path.insert(0, ".")
import cs3_utils as U

ROOT, DATA = U.folders()
CACHE = DATA / "downloads" / "screen"          # shared with notebook 00 (windows table, rMATS tars, chain)
CACHE.mkdir(parents=True, exist_ok=True)
print("RBP-ARK API:", U.RBPARK, "| case study folder:", ROOT, "| data folder:", DATA)

In [ ]:
MAPS = {"QKI": list(U.ECLIP)}
files = U.knockdown_files("differential splicing quantifications")
w = U.rbpark_windows(CACHE, rbps=set(MAPS))
index = U.WindowIndex(w, qmax=0.05, min_l2=2.0)

In [ ]:
rows = []
for rbp, subjects in MAPS.items():
    for cell in ("K562", "HepG2"):
        acc, se = U.se_for(rbp, cell, files, CACHE / "rmats")
        sets = U.event_sets(se)
        for name, ev in sets.items():
            ev = U.lift_events(ev.drop_duplicates(U.SE_COLS[1:5]), CACHE, cols=("exonStart_0base", "exonEnd"))
            for subject in subjects:
                rows.append([f"{rbp}|{cell}", subject, name, len(ev), *U.window_map(ev, index, subject)])
        print(rbp, cell, {k: len(v) for k, v in sets.items()})
counts = pd.DataFrame(rows, columns=["kd", "eclip", "set", "n"] + U.BIN_NAMES)
counts.to_csv(ROOT / "results" / "window_map_counts.csv", index=False)
counts.head()

Expected event counts (exons, after liftover): QKI K562 17 / 10 / 9,523; QKI HepG2 307 / 105 / 9,015 (included / excluded / background).

In [ ]:
tests = []
for (kd_, ec), g in counts.groupby(["kd", "eclip"], sort=False):
    g = g.set_index("set"); bg = g.loc["bg"]
    for s in ("inc", "exc"):
        r = g.loc[s]
        b = max(U.BIN_NAMES, key=lambda x: r[x] / max(r["n"], 1))
        odds, p = fisher_exact([[r[b], r["n"] - r[b]], [bg[b], bg["n"] - bg[b]]])
        tests.append({"knockdown": kd_, "eclip": ec, "set": s, "n": r["n"], "peak bin": b,
                      "% with window": round(100 * r[b] / max(r["n"], 1), 1), "% background": round(100 * bg[b] / bg["n"], 2),
                      "odds ratio": round(odds, 1), "P": p})
tests = pd.DataFrame(tests)
tests.to_csv(ROOT / "results" / "window_map_tests.tsv", sep="\t", index=False)
tests

**Reading it.** QKI: exons more included after knockdown (QKI represses them) have QKI windows just upstream of the 3′ splice site; exons less included (QKI activates them) have windows downstream of the 5′ splice site. With HepG2 knockdown and ENCORE2 eCLIP: 8.5% vs 0.2% (upstream, OR ≈ 42) and 15% vs 0.4% (downstream, OR ≈ 46). Compare with notebook 04's read-based maps.